### Retiro Fugas

In [2]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text


In [3]:

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

In [5]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'


filename='Lista Excepcion 20260730 - Target (002).xlsx'
name_dni='dni'

ruta_archivo = os.path.join(ruta_negocios, filename)
df = pd.read_excel(ruta_archivo)

df[f"{name_dni}"] = (
    df[f"{name_dni}"]
    .astype(str)
    .str.zfill(8)
)

df = df.rename(columns={
    f'{name_dni}': 'NUMDOCUMENTO'
})
df=df[["NUMDOCUMENTO"]]
df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	select NUMDOCUMENTO
	from DANTALION.dbo.Base_Maestra_Efectiva_Negocios_Vigente
    WHERE RETIRO IS NULL OR RETIRO =''
"""
df_tc = pd.read_sql(query, engine_kishin)


In [6]:
df_tc.shape

(144952, 1)

In [10]:
campana='negocios'
query = f"""
    select FECHA as fecha_gestion,DNI as NUMDOCUMENTO,
    PROMOTOR as promotor,
    ESTADO as estado_venta,
    TRAMA_HORA as tramo_venta,
    MONTO as monto_venta,
    DNIEjecutivo as dni_ejecutivo_venta,
    Producto as producto_venta,
    Producto as title,
    Celular as cel_venta,1 as venta,
    subcampana
    from SAMANTHA.dbo.Ventas_Target
    where campana='{campana}'
    and cast(fecha as date) between '{fecha_mes_base}' and EOMONTH('{fecha_mes_base}')
"""
df_ventas = pd.read_sql(query, engine_samantha)

In [8]:
df_ventas.count()

fecha_gestion          7199
NUMERO_DOCUMENTO       7199
promotor               7199
estado_venta           7199
tramo_venta            7199
monto_venta            4667
dni_ejecutivo_venta    7199
producto_venta            0
title                     0
cel_venta              7199
venta                  7199
subcampana                0
dtype: int64

In [11]:
df = df[
    ~df["NUMDOCUMENTO"].isin(df_ventas["NUMDOCUMENTO"])
].copy()


In [12]:
df.merge(df_tc, on='NUMDOCUMENTO', how='inner').count()


NUMDOCUMENTO    66
dtype: int64

In [13]:
df=df.merge(df_tc, on='NUMDOCUMENTO', how='inner')

In [14]:

list_dni = (
    df['NUMDOCUMENTO']
    .dropna()
    .drop_duplicates()
    .tolist()
)
in_clause = ",".join(f"'{x}'" for x in list_dni)

in_clause

"'23717394','40859679','45241705','40189096','40393601','41910742','07273436','41790127','43679764','09065570','07875988','19880437','41235196','42321090','40712590','71666973','46491965','40449738','26723128','60970893','08584643','46586563','47867623','00793370','42714745','41363764','43487400','26655700','71307490','07491863','26610169','44165665','09491639','07956986','27153106','29715387','44591748','08165505','45449800','08885227','41521110','17889369','44052995','29204665','76531569','40904537','46725181','29106652','42558459','43727550','02633339','10263446','10719998','46133136','42919455','44171042','42120852','43792698','42836667','26634825','25802021','17625173','19802329','76669059','05643129','45557162'"

In [ ]:
fecha_mes_base

'2026-07-01'

In [16]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Efectiva_Negocios
            SET RETIRO = 'RETIRO_30_correo'
            WHERE NUMDOCUMENTO IN ({in_clause})
                and fecha_envio >= '{fecha_mes_base}'
                AND fecha_envio <= EOMONTH('{fecha_mes_base}')
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 66


In [26]:
fecha_mes_base

'2026-07-01'

In [ ]:

df_ventas.head()

,fecha_gestion,NUMERO_DOCUMENTO,promotor,estado_venta,tramo_venta,monto_venta,dni_ejecutivo_venta,producto_venta,title,cel_venta,venta,subcampana
0,2026-07-01,07259135,08728081,1.-VALIDADA,9,None,08728081,None,None,981260227,1,None
1,2026-07-01,07779510,44592143,6.-SIN VALIDAR,13,None,44592143,None,None,989371280,1,None
2,2026-07-01,40102384,44592143,1.-VALIDADA,10,None,44592143,None,None,932595948,1,None
3,2026-07-01,40552162,47202133,1.-VALIDADA,10,None,47202133,None,None,988594366,1,None
4,2026-07-01,41266190,44592143,1.-VALIDADA,9,None,44592143,None,None,940049145,1,None


In [17]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Efectiva_Negocios", "SP tNumeros negocios")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Efectiva_Negocios", "SP actualizar negocios Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Efectiva_Negocios", "SP actualizar negocios SA")

SP tNumeros negocios | realizado | duración: 129.37 seg
SP actualizar negocios Zeus | realizado | duración: 219.05 seg
SP actualizar negocios SA | realizado | duración: 10.28 seg


In [8]:
query = f"""
	select NUMERO_DOCUMENTO,PROB_CONTACTO,concat('2026-07-',right(retiro,2)) as retiro
	from DANTALION.dbo.Base_Maestra_Diners_TC_Vigente
    WHERE RETIRO IS not NULL OR RETIRO <>''
"""
df_tc = pd.read_sql(query, engine_kishin)

In [9]:
df_tc.head()

,NUMERO_DOCUMENTO,PROB_CONTACTO,retiro
0,47216152,D,2026-07-01
1,71561197,D,2026-07-01
2,42968074,C,2026-07-01
3,73473064,E,2026-07-01
4,46510882,C,2026-07-01


In [ ]:


server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
fecha_mes_base='2026-06-01'
campana=fecha_a_nombre('2026-05-01')
query = f"""
select dni as NumDoc, 1 as venta_target from SAMANTHA.dbo.Ventas_Target
where CAMPANA='Diners'
and CONVERT(DATE, FECHA) >= CONVERT(DATE, '{fecha_mes_base}')
AND CONVERT(DATE, FECHA) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
   
"""
df_venta = pd.read_sql(query, engine_zeus)

df_target = df_tc.merge(
    df_venta,
    on='NumDoc',
    how='left'
)
df_final = df_target.merge(
    df,
    on='NumDoc',
    how='inner'
)
df_final = (
    df_final[df_final['venta_target'].isnull()]
    .drop(columns=['venta_target'])
)
df_final.rename(
    columns={
        'Importe Solicitado': 'Monto'
    },
    inplace=True
)
print(df_final.columns.tolist())

c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


['NumDoc', 'TIPO_PRODUCTO', 'RETIRO', 'Canal', 'Autor', 'Subcanal', 'Motivo', 'Monto', 'fecha', 'hora']


In [72]:
df_final=df_final[df_final['Monto'].notnull()]

In [78]:
df_final[['TIPO_PRODUCTO','Canal', 'Monto', 'fecha']].head()


,TIPO_PRODUCTO,Canal,Monto,fecha
0,PPD,CANALES DIGITALES,41100.0,2026-06-04
1,PPD,CANALES DIGITALES,13800.0,2026-06-04
2,PPD,CONTACT CENTER,6000.0,2026-06-04


In [ ]:

total_monto_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].sum()

total_ope_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].count()

print(f'PPD | Monto total: {int(total_monto_ppd)} |  Total operaciones {int(total_ope_ppd)} ')

PPD | Monto total: 60900 |  Total operaciones 3 


In [ ]:
# ruta_archivo = os.path.join(ruta_csv, 'PPD_plus.xlsx')
# df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
list_dni = (
    df_final.loc[df_final['RETIRO'].isna(), 'NumDoc']
    .dropna()
    .drop_duplicates()
    .tolist()
)

in_clause = ",".join(f"'{x}'" for x in list_dni)

try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners
            SET RETIRO = 'RETIRO'
            WHERE NumDoc IN ({in_clause})
                and CONVERT(DATE, fecha_envio) >= CONVERT(DATE, '{fecha_mes_base}')
                AND CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)


In [77]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 5.48 seg
SP actualizar diners Zeus | realizado | duración: 5.54 seg
SP actualizar diners SA | realizado | duración: 1.5 seg
